In [ ]:
import numpy as np, torch, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from PIL import Image
from skimage import color, data
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

In [2]:
SIZE, EPOCHS, N_TRAIN, N_VAL = 64, 15, 800, 100
dev = "cuda" if torch.cuda.is_available() else "cpu"
rng = np.random.default_rng(0)

In [ ]:
def to_lab(img):
    """Convert an RGB image to LAB color space and normalize it."""
    lab = color.rgb2lab(np.array(img.convert("RGB").resize((SIZE, SIZE)))).astype("float32")
    return (torch.tensor(lab[..., :1] / 50 - 1).permute(2, 0, 1),
            torch.tensor(lab[..., 1:] / 110).permute(2, 0, 1))

def to_rgb(L, ab):
    """Convert LAB color space back to RGB."""
    lab = torch.cat([(L + 1) * 50, ab * 110], 1).permute(0, 2, 3, 1).detach().cpu().numpy().astype("float64")
    return np.stack([color.lab2rgb(x) for x in lab])

In [ ]:
def make_crops(n, lo, hi):
    """Make random crops from the photos."""
    out = []
    for _ in range(n):
        im = photos[rng.integers(len(photos))]
        h, w = im.shape[:2]; x0, x1 = int(lo * w), int(hi * w)
        s = rng.integers(SIZE, min(h, x1 - x0) + 1)
        y, x = rng.integers(0, h - s + 1), rng.integers(x0, x1 - s + 1)
        out.append(Image.fromarray(im[y:y + s, x:x + s]))
    return out

In [5]:
photos = [data.astronaut(), data.chelsea(), data.coffee(), data.rocket()]

In [ ]:
print("Preparing crops...")

# Create a list of (L, ab) pairs from the crops
pairs = [to_lab(i) for i in tqdm(make_crops(N_TRAIN, 0, .8) + make_crops(N_VAL, .8, 1))]
L_all, ab_all = torch.stack([p[0] for p in pairs]), torch.stack([p[1] for p in pairs])

# Create DataLoaders for training and validation
train_dl = DataLoader(TensorDataset(L_all[:-N_VAL], ab_all[:-N_VAL]), 32, shuffle=True)
val_dl = DataLoader(TensorDataset(L_all[-N_VAL:], ab_all[-N_VAL:]), 32)

In [ ]:
print("Loading pretrained U-Net...")

# Load a pretrained U-Net model with a ResNet34 encoder
model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=1, classes=2, activation="tanh").to(dev)

# Set up the optimizer and loss function
opt, loss_fn = torch.optim.Adam(model.parameters(), 3e-4), torch.nn.L1Loss()

Loading pretrained U-Net...


In [ ]:
for ep in range(EPOCHS):
    # Fine-tune the model on the training data
    model.train()
    for L, ab in tqdm(train_dl, desc=f"Epoch {ep + 1}/{EPOCHS}"):
        L, ab = L.to(dev), ab.to(dev)
        loss = loss_fn(model(L), ab)
        opt.zero_grad(); loss.backward(); opt.step()

Epoch 15/15: 100%|██████████| 25/25 [00:01<00:00, 19.21it/s]


In [ ]:
# Evaluate the model on the validation set and compare it to a grayscale baseline
model.eval()
with torch.no_grad():
    # Compute the mean absolute error for the model and the grayscale baseline
    err_model = err_gray = 0.0
    for L, ab in val_dl:
        L, ab = L.to(dev), ab.to(dev)
        err_model += (model(L) - ab).abs().mean().item() / len(val_dl)
        err_gray += ab.abs().mean().item() / len(val_dl)
    print(f"L1 error on color channels: model = {err_model:.4f} | baseline (grayscale) = {err_gray:.4f}")
    print("Model beats the baseline -> the approach is feasible!" if err_model < err_gray else "Model does not beat the baseline.")

    # Create a comparison figure showing the input grayscale images, the model's predictions, and the ground truth color images
    L, ab = next(iter(val_dl)); L, ab = L[:6].to(dev), ab[:6].to(dev)   # comparison figure
    rows = [to_rgb(L, torch.zeros_like(ab)), to_rgb(L, model(L)), to_rgb(L, ab)]
    fig, axs = plt.subplots(3, 6, figsize=(12, 6))
    for r, name in enumerate(["Input (gray)", "U-Net prediction", "Ground truth"]):
        for c in range(6):
            axs[r, c].imshow(rows[r][c]); axs[r, c].axis("off")
        axs[r, 0].set_title(name, loc="left", fontsize=9)
    plt.tight_layout(); plt.savefig("comparison.png", dpi=120)
    print("Saved comparison.png")

L1 error on color channels: model = 0.0737 | baseline (grayscale) = 0.1152
Model beats the baseline -> the approach is feasible!
Saved comparison.png
